# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# Goal: Build the dataset feature vector for modeling, using the same preprocessing logic as the project scripts.
# This cell loads raw data, standardizes types, fills missing values, and creates engineered features.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository root by walking upward until the project `scripts/` folder is found.
# This allows the notebook to run from the `work/notebooks` folder or another subfolder.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "scripts").exists():
    ROOT = ROOT.parent

if not (ROOT / "scripts").exists():
    raise FileNotFoundError(
        "Cannot find the repository root. Make sure this notebook is inside the project folder."
    )

# Add the repo root to sys.path so Python can import from the local `scripts` package.
sys.path.insert(0, str(ROOT))

from scripts.ml_utils import (
    BOOLEAN_COLUMNS,
    CATEGORICAL_COLUMNS,
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_COLUMNS,
    RAW_PATH,
)

# Load the raw anonymized CSV file from the repository data folder.
raw = pd.read_csv(RAW_PATH)

# Convert numeric columns to numeric dtype and fill missing columns with 0.
for column in NUMERIC_COLUMNS:
    if column in raw.columns:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")
    else:
        raw[column] = 0

# Convert boolean-like columns to actual booleans.
for column in BOOLEAN_COLUMNS:
    if column in raw.columns:
        raw[column] = raw[column].astype(str).str.lower().isin(["true", "1", "yes", "y"])
    else:
        raw[column] = False

# Normalize categorical columns, replacing missing values and blank strings with "unknown".
for column in CATEGORICAL_COLUMNS:
    if column in raw.columns:
        raw[column] = (
            raw[column]
            .fillna("unknown")
            .astype(str)
            .replace({"": "unknown", "nan": "unknown"})
        )
    else:
        raw[column] = "unknown"

# Numeric columns that must be safe for modeling: replace infinities and missing with 0.
numeric_fill_zero = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
]

for column in numeric_fill_zero:
    raw[column] = raw[column].replace([np.inf, -np.inf], np.nan).fillna(0)

# Filter to rows with valid impressions and mature content age.
features = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
features = features.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Create the target label for the decline prediction problem.
features["is_declining_label"] = features["trend_direction"].str.lower().eq("down").astype(int)

# Add transformed and binary engineered features.
features["log_impressions_90d"] = np.log1p(features["impressions_90d"])
features["log_clicks_90d"] = np.log1p(features["clicks_90d"])
features["log_sessions_90d"] = np.log1p(features["sessions_90d"])
features["log_ai_sessions_90d"] = np.log1p(features["ai_sessions_90d"])
features["has_clicks"] = (features["clicks_90d"] > 0).astype(int)
features["has_ai_sessions"] = (features["ai_sessions_90d"] > 0).astype(int)
features["measurable_opportunity"] = (
    (features["impressions_90d"] >= 100) & (features["sessions_90d"] > 0)
).astype(int)

# Define the feature columns used in modeling.
feature_columns = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
print("prepared rows:", len(features))
print("numeric model features:", MODEL_NUMERIC_FEATURES)
print("categorical model features:", MODEL_CATEGORICAL_FEATURES)
features[feature_columns + ["is_declining_label"]].head()

prepared rows: 30000
numeric model features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical model features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,ai_traffic_pct,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,is_declining_label
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,0.0,HIGH,keyword article,transactional,181-365,0-30,2000-3500,good,striking,1
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,0.0,LOW,keyword article,informational,365+,0-30,2000-3500,good,page_3_5,1
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,0.0,LOW,keyword article,informational,91-180,0-30,3500+,good,page_3_5,1
3,10.0,0.00,0.00,0.0,0.0,9.371779,4.077537,4.369448,0.0,88,...,0.0,LOW,keyword article,commercial,365+,0-30,unknown,good,page_1,0
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,0.0,LOW,keyword article,informational,181-365,0-30,2000-3500,good,page_3_5,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# Goal: Describe each model feature and record whether it is available before prediction.
# This cell builds the feature notes table for the model input columns.
feature_notes = {
    "search_volume": (
        "Estimated demand for the content target, filled with 0 when missing. "
        "Available before prediction because it is a prior query-demand signal."
    ),
    "competition": (
        "Numeric competition score, filled with 0 when missing. "
        "Available before prediction as market signal for the query/topic."
    ),
    "cpc": (
        "Cost-per-click estimate, filled with 0 when missing. "
        "Available before prediction from search/advertising data."
    ),
    "word_count": (
        "Content length in words, filled with 0 when missing. "
        "Available before prediction from the page text."
    ),
    "char_count": (
        "Content length in characters, filled with 0 when missing. "
        "Available before prediction from the page text."
    ),
    "log_impressions_90d": (
        "Log-transformed 90-day impressions. Computed after imputing missing impressions with 0. "
        "Available before prediction because it uses historical exposure only."
    ),
    "log_clicks_90d": (
        "Log-transformed 90-day clicks. Computed after imputing missing clicks with 0. "
        "Available before prediction because it uses historical performance only."
    ),
    "log_sessions_90d": (
        "Log-transformed 90-day sessions. Computed after imputing missing sessions with 0. "
        "Available before prediction because it uses historical performance only."
    ),
    "log_ai_sessions_90d": (
        "Log-transformed 90-day AI sessions. Computed after imputing missing AI sessions with 0. "
        "Available before prediction because it uses historical user behavior only."
    ),
    "days_with_impressions": (
        "Count of days with impressions in the lookback window, filled with 0 when missing. "
        "Available before prediction as a prior visibility signal."
    ),
    "days_with_sessions": (
        "Count of days with sessions in the lookback window, filled with 0 when missing. "
        "Available before prediction as a prior engagement signal."
    ),
    "content_age_days": (
        "Age of content in days, filled with 0 when missing. "
        "Available before prediction because it is derived from publish date."
    ),
    "days_since_last_update": (
        "Days since the last content update, filled with 0 when missing. "
        "Available before prediction if update metadata is known."
    ),
    "ctr": (
        "Click-through rate, filled with 0 when missing. "
        "Available before prediction from historical performance."
    ),
    "avg_position": (
        "Average search position, filled with 0 when missing. "
        "Available before prediction from search analytics."
    ),
    "engagement_rate": (
        "Engagement rate on the content, filled with 0 when missing. "
        "Available before prediction from prior behavior."
    ),
    "scroll_rate": (
        "Scroll depth rate, filled with 0 when missing. "
        "Available before prediction from prior user interaction."
    ),
    "ai_traffic_pct": (
        "Share of traffic coming from AI sessions, filled with 0 when missing. "
        "Available before prediction from observed traffic composition."
    ),
    "competition_level": (
        "Categorical competition bucket, unknown is the fallback. "
        "Available before prediction from search market segmentation."
    ),
    "content_type": (
        "Type of content (page/article/video), unknown is the fallback. "
        "Available before prediction from content metadata."
    ),
    "main_intent": (
        "Primary user intent bucket, unknown is the fallback. "
        "Available before prediction from query/content intent classification."
    ),
    "age_tier": (
        "Content age bucket, unknown is the fallback. "
        "Available before prediction from publish-age metadata."
    ),
    "freshness_tier": (
        "Content recency bucket, unknown is the fallback. "
        "Available before prediction from update timestamp metadata."
    ),
    "word_count_tier": (
        "Content length bucket, unknown is the fallback. "
        "Available before prediction from page content."
    ),
    "impression_tier": (
        "Impression volume bucket, unknown is the fallback. "
        "Available before prediction from prior volume history."
    ),
    "position_tier": (
        "Search position bucket, unknown is the fallback. "
        "Available before prediction from ranking analytics."
    ),
}

notes = pd.DataFrame.from_dict(
    {
        feature: {
            "meaning_and_missing": text,
            "available_before_prediction": "yes",
        }
        for feature, text in feature_notes.items()
    }
).reset_index()
notes = notes.rename(columns={"index": "feature"})
notes

,feature,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,...,scroll_rate,ai_traffic_pct,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,meaning_and_missing,"Estimated demand for the content target, fille...","Numeric competition score, filled with 0 when ...","Cost-per-click estimate, filled with 0 when mi...","Content length in words, filled with 0 when mi...","Content length in characters, filled with 0 wh...",Log-transformed 90-day impressions. Computed a...,Log-transformed 90-day clicks. Computed after ...,Log-transformed 90-day sessions. Computed afte...,Log-transformed 90-day AI sessions. Computed a...,...,"Scroll depth rate, filled with 0 when missing....","Share of traffic coming from AI sessions, fill...","Categorical competition bucket, unknown is the...","Type of content (page/article/video), unknown ...","Primary user intent bucket, unknown is the fal...","Content age bucket, unknown is the fallback. A...","Content recency bucket, unknown is the fallbac...","Content length bucket, unknown is the fallback...","Impression volume bucket, unknown is the fallb...","Search position bucket, unknown is the fallbac..."
1,available_before_prediction,yes,yes,yes,yes,yes,yes,yes,yes,yes,...,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [9]:
# Goal: Detect feature leakage by confirming no label-derived columns are included in the model feature list.
# This cell checks each target-related column and raises an error if leakage is found.
import pandas as pd

label_related = ["trend_direction", "trend_pct"]
selected_features = set(feature_columns)
found_leakage = selected_features.intersection(label_related)

print("Label-derived columns present in the selected model features:", found_leakage)

if found_leakage:
    raise AssertionError(
        "Leakage detected: label-derived feature(s) included in the model feature list."
    )
else:
    print("No direct label-derived columns are included in MODEL_NUMERIC_FEATURES or MODEL_CATEGORICAL_FEATURES.")

leakage_check = pd.DataFrame(
    [
        {
            "column": column,
            "in_model_features": column in selected_features,
            "reason": "label/target related"
            if column in label_related
            else "not in current model feature list",
        }
        for column in label_related
    ]
)
leakage_check

Label-derived columns present in the selected model features: set()
No direct label-derived columns are included in MODEL_NUMERIC_FEATURES or MODEL_CATEGORICAL_FEATURES.


,column,in_model_features,reason
0,trend_direction,False,label/target related
1,trend_pct,False,label/target related


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# Goal: Record which raw fields are intentionally excluded from modeling and why.
# This cell lists excluded columns and their reasons to support the leakage/privacy audit.
import pandas as pd

exclusions = [
    {
        "field": "trend_direction",
        "why": "This is the label definition for decline vs. non-decline and must not be used as a feature.",
    },
    {
        "field": "trend_pct",
        "why": "This is derived from the same trend signal and is effectively target leakage, so it is excluded.",
    },
    {
        "field": "provider_used",
        "why": "Product metadata, not a general content signal; it may leak system-specific routing or experiment state.",
    },
    {
        "field": "model_used",
        "why": "Product metadata that is not necessary for a deployable content ranking signal and can overfit to pipeline state.",
    },
]

pd.DataFrame(exclusions)

,field,why
0,trend_direction,This is the label definition for decline vs. n...
1,trend_pct,This is derived from the same trend signal and...
2,provider_used,"Product metadata, not a general content signal..."
3,model_used,Product metadata that is not necessary for a d...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.